# Phase 3 — Run the Bikol NMT API + Web App (Colab, GPU)

This launches your `main.py` (FastAPI) on Colab's GPU and gives you a **public URL** to open the
web interface. It also measures **latency per input length** for the report (criterion C4).

**Before running:** make sure `main.py` and `models/nllb_bikol_lora/` are committed to your repo
(Claude Code did this). Set **Runtime → T4 GPU**, then Run all.

## 1. Install dependencies (pinned to match Phase 2)

In [ ]:
!pip -q install "transformers==4.46.3" "tokenizers==0.20.3" "peft==0.13.2" \
    "accelerate==1.1.1" fastapi "uvicorn[standard]" pydantic requests
print("done")

## 2. Get your code + adapter from GitHub

In [ ]:
!rm -rf bikol-nmt && git clone -q https://github.com/LenardBenavidez/bikol-nmt.git
import os
os.chdir("bikol-nmt")
print("files:", [f for f in os.listdir(".") if not f.startswith(".")])
assert os.path.exists("main.py"), "main.py not found — commit it to the repo first."
assert os.path.isdir("models/nllb_bikol_lora"), "adapter folder missing in repo."
print("OK: main.py and adapter present")

## 3. Start the API server (in the background)

In [ ]:
import subprocess, time, requests, os
# Launch uvicorn serving main.py on port 8000
server = subprocess.Popen(
    ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=open("server.log","w"), stderr=subprocess.STDOUT)
# Wait for it to come up
for _ in range(60):
    try:
        if requests.get("http://localhost:8000/api/v1/health", timeout=2).ok:
            break
    except Exception:
        time.sleep(1)
print(requests.get("http://localhost:8000/api/v1/health").json())

## 4. Get a public URL (cloudflared — no signup)

In [ ]:
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared
import subprocess, re, time
tunnel = subprocess.Popen(["./cloudflared","tunnel","--url","http://localhost:8000","--no-autoupdate"],
                          stdout=open("cf.log","w"), stderr=subprocess.STDOUT)
public_url = None
for _ in range(40):
    time.sleep(1)
    try:
        txt = open("cf.log").read()
        m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", txt)
        if m: public_url = m.group(0); break
    except Exception: pass
print("PUBLIC URL:", public_url)
print("\nOpen that link in a new tab to use the translator UI.")
print("(First translation takes a few seconds while the model loads.)")

### What to do at the public URL (for your screenshots — criterion C2)
Open the link above in a new browser tab. Then capture:
- **Figure 2:** the side-by-side layout with the two language dropdowns
- **Figure 3:** the **Copy** button working
- **Figure 4:** the **Report Inaccurate Translation** button + a submitted report

Type a Bikol sentence, click Translate, and the confidence + latency show under the output.

## 5. Test the API directly (criterion C1 — paste request/response into the report)

In [ ]:
import requests, json
req = {"source_text": "ayoson mo", "source_lang": "bcl_North", "target_lang": "eng_Latn"}
resp = requests.post("http://localhost:8000/api/v1/translate", json=req).json()
print("REQUEST:\n", json.dumps(req, indent=2))
print("\nRESPONSE:\n", json.dumps(resp, indent=2))

## 6. Latency per input length (criterion C4 — the table for your report)

In [ ]:
import requests, time, statistics

# Build inputs of ~10, 25, 50 tokens by repeating a phrase.
base = "ayoson mo an makina ta dai na ini naggagana kaya kaipuhan tang usipon"
samples = {10: " ".join(base.split()[:10]),
           25: " ".join((base.split()*3)[:25]),
           50: " ".join((base.split()*5)[:50])}

print(f"{'tokens':>7} | {'avg latency (ms)':>16} | {'<=300ms?':>8}")
print("-"*40)
for ntok, text in samples.items():
    times = []
    for _ in range(5):                      # 5 runs, average
        t0 = time.perf_counter()
        requests.post("http://localhost:8000/api/v1/translate",
                      json={"source_text": text, "source_lang":"bcl_North", "target_lang":"eng_Latn"})
        times.append((time.perf_counter()-t0)*1000)
    avg = statistics.mean(times)
    print(f"{ntok:>7} | {avg:>16.1f} | {'yes' if avg<=300 else 'no':>8}")
print("\nHardware: Colab T4 GPU. Timer measures end-to-end request (model already loaded).")

## 7. Check the feedback log (criterion C3)
After you submit at least one report from the web UI, run this to show a stored row.

In [ ]:
import sqlite3, pandas as pd
con = sqlite3.connect("results/feedback.db")
try:
    df = pd.read_sql_query("SELECT * FROM feedback ORDER BY id DESC LIMIT 5", con)
    print("rows stored:", len(pd.read_sql_query("SELECT id FROM feedback", con)))
    display(df)
except Exception as e:
    print("No reports yet — submit one from the web UI first.", e)
con.close()

---
### For the report (Section 4, criteria C1–C4)
- **C1 endpoint:** `POST /api/v1/translate` — request/response from cell 5.
- **C2 web interface:** screenshots from the public URL (Figures 2–4).
- **C3 feedback logging:** SQLite at `results/feedback.db`; example row from cell 7.
- **C4 latency:** table from cell 6 (Colab T4 GPU, avg of 5 runs per length).